In [6]:
import unicodedata

import altair as alt
import pandas as pd
from PIL import Image

RUTA = "base_datos_oficial_ordenada.csv"  # CSV separado por comas

In [7]:
COLORES = {
    "Romance":  "#E75A9B",  # rosado
    "Comedia":  "#F28E2B",  # naranja
    "Bélica":   "#D62728",  # rojo
    "Historia": "#8C564B",  # café
    "Crimen":   "#1F77B4",  # azul
    "Suspenso": "#7B4FA8",  # morado
    "Drama":    "#7F7F7F",  # gris oscuro
    "Otros":    "#BDBDBD",  # gris claro
}
ORDEN = list(COLORES)

In [8]:
MAPA_GENEROS = {
    "romance": "Romance",
    "comedy": "Comedia", "comedia": "Comedia",
    "war": "Bélica", "belica": "Bélica", "guerra": "Bélica",
    "history": "Historia", "historia": "Historia",
    "crime": "Crimen", "crimen": "Crimen",
    "thriller": "Suspenso", "suspense": "Suspenso", "suspenso": "Suspenso",
    "drama": "Drama",
}

In [9]:
MAPA_GENEROS = {
    "romance": "Romance",
    "comedy": "Comedia", "comedia": "Comedia",
    "war": "Bélica", "belica": "Bélica", "guerra": "Bélica",
    "history": "Historia", "historia": "Historia",
    "crime": "Crimen", "crimen": "Crimen",
    "thriller": "Suspenso", "suspense": "Suspenso", "suspenso": "Suspenso",
    "drama": "Drama",
}

In [11]:
ANGLO = {
    "estadounidense", "britanica", "canadiense", "australiana", "irlandesa",
    "neozelandesa", "neozelandes", "norirlandesa", "sudafricana",
    "escocesa", "galesa", "inglesa",
}

def sin_tildes(texto):
    texto = unicodedata.normalize("NFD", str(texto).strip().lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

In [12]:
def cargar():
    df = pd.read_csv(RUTA, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    df["nac"] = df["nacionalidad_1"].map(sin_tildes)
    df["grupo"] = None
    df.loc[df["nac"] == "estadounidense", "grupo"] = "Estadounidenses"
    df.loc[~df["nac"].isin(ANGLO), "grupo"] = "Extranjeros (no anglosajones)"
    return df

In [13]:
def generos_de(celda):
    """Convierte 'Drama, Romance' en {'Drama', 'Romance'} con nombres estándar."""
    cat = set()
    for g in str(celda).split(","):
        g = sin_tildes(g)
        if g and g != "nan":
            cat.add(MAPA_GENEROS.get(g, "Otros"))
    return cat

In [14]:
def conteo(sub):
    """% de nominaciones del grupo cuya película tiene cada género."""
    filas = []
    for celda in sub["generos"]:
        for g in generos_de(celda):
            filas.append(g)
    n = len(sub)
    tabla = pd.Series(filas).value_counts().reindex(ORDEN, fill_value=0)
    return pd.DataFrame({
        "genero": tabla.index,
        "nominaciones": tabla.values,
        "porcentaje": tabla.values / n,
    })

In [15]:
def grafico(sub, titulo, con_leyenda):
    d = conteo(sub)
    escala = alt.Scale(domain=ORDEN, range=[COLORES[g] for g in ORDEN])
    return (
        alt.Chart(d)
        .mark_bar()
        .encode(
            y=alt.Y("genero:N", sort=ORDEN, title=None),
            x=alt.X("porcentaje:Q", title="% de las nominaciones",
                    axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 1])),
            color=alt.Color(
                "genero:N", scale=escala,
                legend=alt.Legend(title="Género") if con_leyenda else None,
            ),
            tooltip=[
                alt.Tooltip("genero:N", title="Género"),
                alt.Tooltip("nominaciones:Q", title="Nominaciones"),
                alt.Tooltip("porcentaje:Q", title="% del grupo", format=".0%"),
            ],
        )
        .properties(title=f"{titulo} ({len(sub)} nominaciones)", width=320, height=260)
    )

In [18]:
if __name__ == "__main__":
    # Intentar importar vl-convert; si falla, instalarlo automáticamente
    try:
        import vl_convert as vlc
    except ImportError:
        import sys
        import subprocess
        print("Instalando vl-convert-python para exportar imágenes estáticas...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "vl-convert-python"])

    df = cargar()
    ext = df[df["grupo"] == "Extranjeros (no anglosajones)"]
    usa = df[df["grupo"] == "Estadounidenses"]

    chart = (
        alt.hconcat(
            grafico(ext, "Extranjeros", con_leyenda=False),
            grafico(usa, "Estadounidenses", con_leyenda=True),
        )
        .resolve_scale(color="shared")
        .properties(title="Géneros de las películas nominadas a mejor actor y actriz")
        .configure_view(stroke=None)
    )

    chart.save("grafico_generos.html")
    chart.save("grafico_generos.png", scale_factor=2)  # ahora funcionará con vl-convert-python instalado
    Image.open("grafico_generos.png").convert("RGB").save("grafico_generos.jpg", quality=95)
    print("Listo: grafico_generos.html y grafico_generos.jpg")

Listo: grafico_generos.html y grafico_generos.jpg


In [19]:
from google.colab import files
import os

# Descargar los archivos generados si existen
for archivo in ["grafico_generos.html", "grafico_generos.jpg"]:
    if os.path.exists(archivo):
        print(f"Descargando {archivo}...")
        files.download(archivo)
    else:
        print(f"El archivo {archivo} no se encontró en el directorio actual.")

Descargando grafico_generos.html...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando grafico_generos.jpg...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>